# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Dataset Citation: {getattr(meta, 'cite_as', getattr(meta, 'citeAs', None))}")
print(f"License: {meta.license}")
print(f"Published: {meta.date_published if hasattr(meta, 'date_published') else getattr(meta, 'datePublished', '-')}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All identifiers are given by `@id` fields.

Note: You can see the available record sets, each of which will have a unique `@id`.

In [ ]:
# Show available record sets and their information
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets and Their Fields:")
record_set_ids = []
for recset in record_sets:
    print(f"- RecordSet: {recset['@id']}")
    record_set_ids.append(recset['@id'])
    # List all fields for this record set
    fields = recset.get('fields', [])
    if not fields:
        print('    No fields found in this record set.')
    else:
        print('    Fields:')
        for field in fields:
            if isinstance(field, str):  # may be only @id string
                print(f"      - {field}")
            elif isinstance(field, dict):
                f_id = field.get('@id', None)
                f_name = field.get('name', '')
                print(f"      - {f_id} ({f_name})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference record set and field `@id`s from the overview above.

**Example:**

`data = dataset.records(record_set='@id-of-recordset')`

In [ ]:
# Extract data from all record sets
import pprint
dataframes = {}

for recset_id in record_set_ids:
    print(f"Loading records from {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    if len(records) == 0:
        print(f"  [No records found for {recset_id}]")
        continue
    # Store as DataFrame
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    # Show field names
    print(f"  Fields (@id): {df.columns.tolist()}")
    print()
# Pick the first available record set for demonstration
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"First available RecordSet ID for analysis: {main_record_set_id}\n")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All field columns refer to the `@id` as the column name.


In [ ]:
# Explore numeric fields in the first record set
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields (@id): {numeric_fields}")

# If numeric fields found, pick one for analysis
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Working with numeric field: {numeric_field}\n")
    # Filter: values > a threshold (median)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
    group_field = None
    for col in non_numeric_fields:
        n_unique = df[col].nunique()
        if n_unique < max(10, 0.2 * len(df)):
            group_field = col
            break
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())
    else:
        print("No suitable grouping field found.")
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or relationship between two fields in the dataset (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a second numeric field exists, make a scatterplot
    if len(numeric_fields) > 1:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("No numeric fields for visualization.")

## 6. Conclusion
In this notebook, we have loaded the FAIR<sup>2</sup> Mass Spectrometry dataset, reviewed its schema, loaded records by their `@id` references, performed basic exploratory data analysis on its numeric fields (filtering, normalization, grouping), and visualized the data distribution.

- All references to record sets and field columns are performed using their `@id`.
- Processing and visualization code can be adapted per the actual field structure and quantity.
- Use this workflow to further analyze, filter, and model the dataset for your downstream applications.